# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('..')
from src.data_cleaning import filter_target_crops, remove_invalid_rows, handle_missing_values, cap_rainfall_outliers
from src.feature_engineering import create_yield_per_hectare
from src.scraping import fetch_wikipedia_tables
from src.merge_data import merge_supplementary_data

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline


In [18]:
# Load the dataset from Phase 2 (update the path as needed)
# DATA_PATH = 'path/to/your/data.csv'

# df = pd.read_csv(DATA_PATH)
# print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
# df.head()

DATA_PATH = '../data/raw/crop_yield.csv'
try:
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
    display(df.head())
except FileNotFounderror:
    print('error: Dataset not found. Please ensure crop_yield.csv is located in the data/raw/ directory.')


Loaded dataset: 19689 rows x 10 columns


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [19]:
# Focus analysis exclusively on Rice and Wheat crops using src function
df_selected = filter_target_crops(df)
print(f'Shape after filtering for Rice and Wheat(rows,columns): {df_selected.shape}')


Shape after filtering for Rice and Wheat(rows,columns): (1742, 7)


In [ ]:
# Remove anomalous records using src function
df_selected = remove_invalid_rows(df_selected)
print(f'Shape after removing invalid records(rows,columns): {df_selected.shape}')


Shape after removing invalid records: (1742, 7)


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [21]:
print('Missing values before cleaning:')
print(df_selected.isnull().sum())

# Handle missing values using src function
df_clean = handle_missing_values(df_selected)

print(f'Missing values remaining: {df_clean.isnull().sum().sum()}')


Missing values before cleaning:
State         0
Crop          0
Crop_Year     0
Season        0
Area          0
Production    0
Fertilizer    0
dtype: int64
Missing values remaining: 0


In [22]:
# TODO: Remove duplicate records.

# before = len(df_clean)
# df_clean = df_clean.drop_duplicates()
# after = len(df_clean)
# print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")

before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")


Removed 0 duplicate rows. Remaining: 1742 rows.


In [23]:
# Cap extreme rainfall outliers using src function
df_clean = cap_rainfall_outliers(df_clean)


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [24]:
# core project requirement: Create the yield_per_hectare target variable using src function
df_clean = create_yield_per_hectare(df_clean)
display(df_clean[['Crop', 'Production', 'Area', 'yield_per_hectare']].head())


Feature engineering complete: yield_per_hectare added.


,Crop,Production,Area,yield_per_hectare
16,Rice,398311,607358.0,0.655809
17,Rice,209623,174974.0,1.198024
18,Rice,1647296,1743321.0,0.944918
26,Wheat,110054,84698.0,1.299370
51,Rice,2340493,1031530.0,2.268953


In [ ]:

# check data shape unchanged need to check over merging and scraping !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# core project requirement: Web Scraping
# Scrape the Wikipedia table and then merge into primary dataset
df_wiki = fetch_wikipedia_tables()
display(df_wiki.head() if not df_wiki.empty else 'Failed to scrape')

df_integrated = merge_supplementary_data(df_clean, df_wiki)
print(f'Integrated dataset shape: {df_integrated.shape}')


Scraping successful. Extracted 29 states/regions from Wikipedia.


,State,Zone
0,Andhra Pradesh,Southern
1,Arunachal Pradesh,North-Eastern
2,Assam,North-Eastern
3,Bihar,Eastern
4,Chhattisgarh,Central


Merging primary dataset with scraped Wikipedia data on [State]...
Merge complete. New dataset shape: (1742, 9)
Integrated dataset shape: (1742, 9)


In [26]:
# TODO: Encode categorical variables.
# Example: One-hot encoding
# df_integrated = pd.get_dummies(df_integrated, columns=['categorical_col_1', 'categorical_col_2'], drop_first=True)
# Example: Label encoding for ordinal variables
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# df_integrated['ordinal_col'] = le.fit_transform(df_integrated['ordinal_col'])
# Encode categorical variables using one-hot encoding for analytical compatibility
# We encode Crop and Season to classify the agricultural cycle linearly.
categorical_columns = ['Crop', 'Season', 'State', 'Zone']
existing_cats = [col for col in categorical_columns if col in df_integrated.columns]
df_integrated = pd.get_dummies(df_integrated, columns=existing_cats, drop_first=True)
print('Categorical variables successfully encoded.')


Categorical variables successfully encoded.


In [27]:
# TODO: Scale / normalise numerical features if required.
# from sklearn.preprocessing import StandardScaler, MinMaxScaler
# scaler = StandardScaler()  # or MinMaxScaler()
# cols_to_scale = ['feature_1', 'feature_2']
# df_integrated[cols_to_scale] = scaler.fit_transform(df_integrated[cols_to_scale])
# Standardize numerical features to ensure predictive algorithms process them equally
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
cols_to_scale = ['Area', 'Production', 'Rainfall', 'Temperature', 'Fertilizer']
valid_cols_to_scale = [col for col in cols_to_scale if col in df_integrated.columns]
df_integrated[valid_cols_to_scale] = scaler.fit_transform(df_integrated[valid_cols_to_scale])
print('Numerical features successfully standardized.')


Numerical features successfully standardized.


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [28]:
# Optional: Verify the integrated data
# df_integrated.head()

---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [29]:
# TODO: Apply final formatting — data types, column order, renaming.

# Example: Convert data types
# df_integrated['some_column'] = df_integrated['some_column'].astype('int')

# Example: Rename columns for clarity
# df_integrated = df_integrated.rename(columns={
#     'old_name': 'new_descriptive_name'
# })

# Example: Reorder columns (target column last)
# target_col = 'target'
# feature_cols = [col for col in df_integrated.columns if col != target_col]
# df_final = df_integrated[feature_cols + [target_col]]

target_col = 'yield_per_hectare'
# Place target column at the very end
feature_cols = [col for col in df_integrated.columns if col != target_col]
df_final = df_integrated[feature_cols + [target_col]]


In [32]:
# TODO: Verify the final prepared dataset.

# print("=" * 60)
# print("FINAL PREPARED DATASET SUMMARY")
# print("=" * 60)
# print(f"Shape: {df_final.shape}")
# print(f"Missing values: {df_final.isnull().sum().sum()}")
# print(f"Duplicates: {df_final.duplicated().sum()}")
# print(f"\nColumn types:")
# print(df_final.dtypes)
# df_final.head()

print('=' * 60)
print('FINAL PREPARED DATASET SUMMARY')
print('=' * 60)
print(f"Final Shape: {df_final.shape}")
display(df_final.head(10))


FINAL PREPARED DATASET SUMMARY
Final Shape: (1742, 45)


,Crop_Year,Area,Production,Fertilizer,Crop_Wheat,Season_Kharif,Season_Rabi,Season_Summer,Season_Whole Year,Season_Winter,State_Arunachal Pradesh,State_Assam,State_Bihar,State_Chhattisgarh,State_Delhi,State_Goa,State_Gujarat,State_Haryana,State_Himachal Pradesh,State_Jammu and Kashmir,State_Jharkhand,State_Karnataka,State_Kerala,State_Madhya Pradesh,State_Maharashtra,State_Manipur,State_Meghalaya,State_Mizoram,State_Nagaland,State_Odisha,State_Puducherry,State_Punjab,State_Sikkim,State_Tamil Nadu,State_Telangana,State_Tripura,State_Uttar Pradesh,State_Uttarakhand,State_West Bengal,Zone_Eastern,Zone_North-Eastern,Zone_Northern,Zone_Southern,Zone_Western,yield_per_hectare
0,1997,-0.188301,-0.409233,-0.288124,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,0.655809
1,1997,-0.452854,-0.449431,-0.469397,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,1.198024
2,1997,0.506735,-0.143145,0.188118,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,0.944918
3,1997,-0.508089,-0.470644,-0.507244,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,1.299370
4,1997,0.071227,0.004535,-0.110294,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,2.268953
5,1997,-0.526940,-0.470794,-0.520160,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,2.029171
6,1997,-0.395910,-0.331590,-0.430378,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,2.845648
7,1997,-0.406523,-0.468842,-0.437650,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,0.472728
8,1997,-0.495564,-0.462112,-0.498661,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,1.427223
9,1997,-0.557289,-0.492621,-0.540955,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,1.608118


In [31]:
# TODO: Save the prepared dataset for use in Phase 4 (Modelling).

# OUTPUT_PATH = 'path/to/your/prepared_data.csv'
# df_final.to_csv(OUTPUT_PATH, index=False)
# print(f"Prepared dataset saved to: {OUTPUT_PATH}")

import os
os.makedirs('../data/processed', exist_ok=True)
OUTPUT_PATH = '../data/processed/crop_yield_cleaned.csv'
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Prepared dataset saved to: {OUTPUT_PATH}")


Prepared dataset saved to: ../data/processed/crop_yield_cleaned.csv
